## IDA Outline
- Load and clean on the full dataset (no learning from price)
- Check shape, dtypes, date range, and rows per month.
- Fix types and units. Parse dates, cast numerics, and standardize units (sqft, lot size in acres vs. sqft).
- Standardize categories. Normalize labels (e.g., "Single Family" vs "SFR") and strip whitespace and case differences.
- Remove duplicates. Decide how to handle relists and repeat sales (keep the final sale, or keep both with a flag).
- Fix impossible values. Drop or null out rows like $0/$1 prices, 0 sqft, future sale dates, or a year built of 1800.
- Profile missingness. Compute null % per column overall and per month or quarter.
- Drop columns. Remove columns that are all-null, constant, or ID-like, plus columns that won't exist at prediction time (e.g., final sale price components, days on market if it's only known after sale).
- Check for target leakage. Look for any column derived from or only known after the sale.

### Reasons for analysis:
- Understand the data before building a ML model
- Analysis to uncover interesting patterns

#### Loading

In [1]:
# Install required packages once in this notebook environment
# python -m 
# import pip
# %pip install pandas matplotlib seaborn numpy

In [ ]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date
# import seaborn as sns   # numpy error, check version and env

In [ ]:
# Concat the data from multiple sources to make one complete dataset
# list of files to open
files = ["//CRML_Data"]

# read and concatenate all files into a single DataFrame
folder = Path(r"C:\Users\Owner\Documents\IDX\CRML_Data")
files = sorted(folder.glob("*.csv"))  # reads all CSVs in the folder

df_list = [pd.read_csv(file) for file in files]
df = pd.concat(df_list, ignore_index=True)

# turn off truncation of DataFrame display and printing, since we know that there are over two dozen
# separate files, and over 70 different columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

C:\Users\Owner\AppData\Local\Temp\ipykernel_21088\2119264079.py:9: DtypeWarning: Columns (2: WaterfrontYN, 36: ElementarySchool, 39: BuilderName, 56: CoBuyerAgentFirstName, 74: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in files]
C:\Users\Owner\AppData\Local\Temp\ipykernel_21088\2119264079.py:9: DtypeWarning: Columns (4: WaterfrontYN) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in files]
C:\Users\Owner\AppData\Local\Temp\ipykernel_21088\2119264079.py:9: DtypeWarning: Columns (4: WaterfrontYN, 74: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(file) for file in files]


#### Filtering for specified parameters

In [ ]:
# Apply known filters, 'StateOrProvince' only == 'CA', 
# 'PropertyType' column to only include 'Residential',
# 'PropertySubtype' to SingleFamilyResidence'  
df = df[df['PropertyType'].isin(['Residential'])]
df = df[df['PropertySubType'].isin(['SingleFamilyResidence'])]
df = df[df['StateOrProvince'] == 'CA']

df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,BuyerOfficeName,CoListOfficeName,ListAgentFullName,CoListAgentFirstName,CoListAgentLastName,BuyerAgentMlsId,BuyerAgentFirstName,BuyerAgentLastName,FireplacesTotal,AssociationFeeFrequency,AboveGradeFinishedArea,ListingKeyNumeric,MLSAreaMajor,TaxAnnualAmount,CountyOrParish,MlsStatus,ElementarySchool,AttachedGarageYN,ParkingTotal,BuilderName,PropertySubType,LotSizeAcres,SubdivisionName,BuyerOfficeAOR,YearBuilt,BuyerAgencyCompensationType,StreetNumberNumeric,ListingId,BathroomsTotalInteger,City,BuyerAgencyCompensation,TaxYear,BuildingAreaTotal,BedroomsTotal,ContractStatusChangeDate,ElementarySchoolDistrict,CoBuyerAgentFirstName,PurchaseContractDate,ListingContractDate,BelowGradeFinishedArea,BusinessType,StateOrProvince,CoveredSpaces,MiddleOrJuniorSchool,FireplaceYN,Stories,HighSchool,Levels,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName
5,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,Michael,Darwich,32.659315,-117.096922,2811 C Avenue,Residential,1974.0,759900.0,33,Berkshire Hathaway HomeServices California Pro...,FOSTER HAMILTON Real Estate,NaN,Michael Darwich,NaN,NaN,SAND-606727,Charles,Street,NaN,NaN,NaN,522107581,91950 - National City,NaN,San Diego,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,NaN,National City,SanDiego,2023.0,Item1,2811.0,210014555,4.0,National City,2.0,NaN,NaN,4.0,2024-01-05,NaN,NaN,2021-06-30,2021-03-08,NaN,NaN,CA,NaN,NaN,False,2.0,NaN,Two,NaN,NaN,NaN,False,2.0,NaN,91950,NaN,NaN,NaN,True,True,NaN,NaN,NaN,NaN
9,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,Michael,Darwich,32.659284,-117.097174,2812 C Avenue,Residential,1974.0,770000.0,228,Berkshire Hathaway HomeServices California Pro...,FOSTER HAMILTON Real Estate,NaN,Michael Darwich,NaN,NaN,SAND-606727,Charles,Street,NaN,NaN,NaN,510919001,91950 - National City,NaN,San Diego,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,NaN,National City,SanDiego,2023.0,Item1,2812.0,210008330,4.0,National City,2.0,NaN,NaN,4.0,2024-01-05,NaN,NaN,2021-11-18,2021-03-08,NaN,NaN,CA,NaN,NaN,False,2.0,NaN,Two,NaN,NaN,NaN,False,2.0,NaN,91950,NaN,NaN,NaN,True,True,NaN,NaN,NaN,NaN
28,NaN,True,NaN,NaN,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,Jennifer,Kennedy,35.277199,-120.646786,2054 Fixlini Street,Residential,3736.0,2100000.0,0,Home & Ranch Sotheby's Intl,Home & Ranch Sotheby's Intl,NaN,Jennifer Kennedy,NaN,NaN,NS02093802,Jennifer,Kennedy,NaN,NaN,NaN,1067652762,SLO - San Luis Obispo,NaN,San Luis Obispo,Closed,NaN,True,3.0,NaN,SingleFamilyResidence,0.2576,San Luis Obispo(380),NorthSanLuisObispo,2019.0,Item1,2054.0,NS24001417,4.0,San Luis Obispo,3.0,NaN,NaN,4.0,2024-01-02,NaN,NaN,2023-11-15,2023-11-15,NaN,NaN,CA,NaN,NaN,True,2.0,NaN,Two,NaN,11219.00,3.0,False,3.0,San Luis Coastal Unified,93401,0.0,11219.0,NaN,False,False,NaN,NaN,NaN,NaN
29,Tile,True,NaN,NaN,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,Kira,Meade,39.419080,-123.814842,31451 Bay View Avenue,Residential,2100.0,1950000.0,-36,Mendo Sotheby's International Realty,West Coast Property Management and Sales,NaN,Kira Meade,NaN,NaN,VCR-cmdeedeethomas,DeeDee,Thomas,NaN,NaN,NaN,1063453216,FBS - Fort Bragg South,NaN,Mendocino,Closed,NaN,True,2.0,NaN,SingleFamilyResidence,1.7100,NaN,CoastalMendocino,1986.0,Item1,31451.0,C1-10456,0.0,Fort Bragg,2.5,NaN,NaN,3.0,2024-01-22,NaN,NaN,2023-12-19,2023-11-17,NaN,NaN,CA,NaN,NaN,True,2.0,NaN,Two,NaN,1.71,NaN,NaN,2.0,NaN,95437,NaN,74487.6,NaN,False,False,NaN,NaN,NaN,NaN
31,"Tile,Wood",True,NaN,NaN,Na

#### Missingness 
Report missingness by column w/out dropping.

In [5]:
# 1b. Need to explore NaNs 
# This step involves checking for missing values in the dataset to understand the extent of incomplete data and plan for 
# handling them appropriately. How many are there, and in which columns? Which columns can be dropped to improve data quality?
print("Number of columns with NaNs:", df.isnull().any().sum().round(2))

# count percentage of NaNs per column with NaNs
# limit decimals to 2 for better readability
# sort by percentage of NaNs in descending order to identify columns with the most missing values
print("Percentage of NaNs per column:", (df.isnull().mean() * 100).round(2).sort_values(ascending=False))

Number of columns with NaNs: 70
Percentage of NaNs per column: TaxYear                         100.00
FireplacesTotal                 100.00
TaxAnnualAmount                 100.00
AboveGradeFinishedArea          100.00
ElementarySchoolDistrict        100.00
BusinessType                    100.00
CoveredSpaces                   100.00
MiddleOrJuniorSchoolDistrict    100.00
WaterfrontYN                     99.96
BelowGradeFinishedArea           99.39
BasementYN                       97.59
BuilderName                      95.33
LotSizeDimensions                93.88
BuildingAreaTotal                93.65
CoBuyerAgentFirstName            91.01
OriginatingSystemName            89.34
OriginatingSystemSubName         89.34
BuyerAgencyCompensationType      86.36
BuyerAgencyCompensation          86.36
ElementarySchool                 86.32
MiddleOrJuniorSchool             86.21
HighSchool                       82.11
CoListAgentFirstName             77.19
CoListAgentLastName              77.12
C

In [8]:
df.isnull().sum()

Flooring                        110710
ViewYN                           27611
WaterfrontYN                    309577
BasementYN                      302259
PoolPrivateYN                    29791
OriginalListPrice                  607
ListingKey                           0
ListAgentEmail                     738
CloseDate                            0
ClosePrice                           2
ListAgentFirstName                2480
ListAgentLastName                   28
Latitude                          2712
Longitude                         2712
UnparsedAddress                    294
PropertyType                         0
LivingArea                         164
ListPrice                            0
DaysOnMarket                         0
ListOfficeName                       0
BuyerOfficeName                   4894
CoListOfficeName                236708
ListAgentFullName                   60
CoListAgentFirstName            239072
CoListAgentLastName             238865
BuyerAgentMlsId          

Of the original 84 columns, only 46 remain after NaN removal for columns with > 10% NaNs; a ~50% reduction of the total dataframe size. We still need to checked the difference between files marked with '_filled' vs the rest of the files.

UPDATE: 'latfilled' and 'lonfilled' are the 2 extra columns in the _filled csvs. 

#### Duplicates
Report exact duplicate rows w/out automatically deleting them.

In [11]:
# need to check for duplicates, use ListingKey as the unique identifier for each listing, and check for duplicates in that column
# need to print to see the duplicates
# change to see only duplicates, not the entire DataFrame
duplicates = df['ListingKey'].duplicated()
print(df.loc[duplicates])

                               Flooring ViewYN WaterfrontYN BasementYN  \
95727                               NaN   True          NaN        NaN   
99544                               NaN  False          NaN        NaN   
102095                        Tile,Wood   True          NaN        NaN   
103244                   Laminate,Stone   True          NaN        NaN   
105553                              NaN   True          NaN        NaN   
106379                         Laminate    NaN          NaN        NaN   
106524                              NaN   True          NaN        NaN   
107485             Carpet,Laminate,Tile  False          NaN        NaN   
107794                              NaN  False          NaN        NaN   
108299                    Laminate,Tile   True          NaN        NaN   
108329                         Laminate   True          NaN        NaN   
108591                         Laminate  False          NaN        NaN   
109120                            Viny

We'll work on converting the dtypes, and addressing the mismatched types mentioned during loading.

In [12]:
# Convert columns to appropriate data types
bool_columns = ['ViewYN', 'FireplaceYN', 'NewConstructionYN', 'PoolPrivateYN']
# how to fill these NAs?
for col in bool_columns:
    df[col] = df[col].astype('boolean', errors='ignore')
    #print(f"Column '{col}' dtype: {df[col].dtype}")

In [13]:
# convert relevant cols to int64
int_columns = ['BedroomsTotal', 'BathroomsTotalInteger', 'YearBuilt',
               'ListingKeyNumeric', 'BuyerAgentMlsId', 'DaysOnMarket', 'ListPrice',
                'LivingArea', 'ClosePrice', 'OriginalListPrice']
for col in int_columns:
    df[col] = df[col].astype('Int64', errors='ignore')
    #print(f"Column '{col}' dtype: {df[col].dtype}")

#### Inspect date parsing, date ranges, invalid values, and numeric ranges.

In [14]:
# check date ranges for each date column and convert to datetime format
date_columns = ['ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'CloseDate']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"Column '{col}' date range: {df[col].min()} to {df[col].max()}")

Column 'ContractStatusChangeDate' date range: 2024-01-01 00:00:00 to 2026-04-30 00:00:00
Column 'PurchaseContractDate' date range: 2020-08-18 00:00:00 to 2026-04-30 00:00:00
Column 'ListingContractDate' date range: 1984-08-28 00:00:00 to 2026-05-01 00:00:00
Column 'CloseDate' date range: 2024-01-01 00:00:00 to 2026-04-30 00:00:00


In [15]:
df['PostalCode'] = df['PostalCode'].astype(str)

# Categories
categorical_vars = ['ListAgentEmail', 'PropertyType', 'ListOfficeName', 'BuyerOfficeName',
       'BuyerAgentFirstName', 'BuyerAgentLastName', 'CountyOrParish', 'MlsStatus', 
       'PropertySubType', 'BuyerOfficeAOR', 'City', 'StateOrProvince', 'PostalCode']

cat_summary = pd.DataFrame({
    'n_unique': df[categorical_vars].nunique(),
    'n_missing': df[categorical_vars].isna().sum(),
    'pct_missing': (df[categorical_vars].isna().mean() * 100).round(1),
    'top_value': df[categorical_vars].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None),
}).sort_values('n_unique')

print(cat_summary)

                     n_unique  n_missing  pct_missing  \
PropertyType                1          0          0.0   
MlsStatus                   1          0          0.0   
StateOrProvince             1          0          0.0   
PropertySubType             1          0          0.0   
CountyOrParish             63          0          0.0   
BuyerOfficeAOR             68      17145          5.5   
City                     1101        240          0.1   
PostalCode               3174          1          0.0   
ListOfficeName          16694          0          0.0   
BuyerAgentFirstName     17321       1594          0.5   
BuyerOfficeName         19009       4894          1.6   
BuyerAgentLastName      36032        131          0.0   
ListAgentEmail          72387        738          0.2   

                                                    top_value  
PropertyType                                      Residential  
MlsStatus                                              Closed  
StateOrPr

In [16]:
MAX_FULL = 25   # show every category if a column has this many unique values or fewer
TOP_N = 10      # otherwise show only the most frequent

for var in categorical_vars:
    n_unique = df[var].nunique()
    counts = df[var].value_counts(dropna=False)
    
    print(f"\n{var} ({n_unique} unique)")
    if n_unique <= MAX_FULL:
        print(counts)
    else:
        print(f"Top {TOP_N}:")
        print(counts.head(TOP_N))


ListAgentEmail (72387 unique)
Top 10:
ListAgentEmail
randy.anderson@richmondamericanhomes.com    817
homes@opendoor.com                          798
NaN                                         738
pxmartinez@kbhome.com                       714
pexanthakis@drhorton.com                    517
lolivo@taylormorrison.com                   424
mhandler@drhorton.com                       413
info@homecoin.com                           373
sold@myattrealestate.com                    364
cesi@cesipagano.com                         354
Name: count, dtype: int64

PropertyType (1 unique)
PropertyType
Residential    309713
Name: count, dtype: int64

ListOfficeName (16694 unique)
Top 10:
ListOfficeName
Compass                                                  20825
Coldwell Banker Realty                                   13043
Keller Williams Realty                                    5785
First Team Real Estate                                    4307
Berkshire Hathaway HomeServices California Prope

### Observations

It looks like there are some data discrepanices. The data in the file is supposed to be for one month, however, this shows records going back to 1984. This could be due to the fact that a contract or listing could have initially occurred much longer ago. ContractStatusChangeDate and CloseDate seem to be correlated, which makes sense, as the final sale of a property would change its listing status.

For our analysis, it seems that if we include these variables, we should assign date limiters. Remove columns that are all-null, constant, or ID-like, plus columns that won't exist at prediction time (e.g., final sale price components, days on market if it's only known after sale). Check for target leakage. Look for any column derived from or only known after the sale.

In [ ]:
# split data to training, validation, and test sets using CloseDate
# training date range: up to 2026-03
# validation date range: 2026-04 to 2026-05
# test date range: 2026-06 and beyond
df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')

# Split the data based on CloseDate
train_df = df[df['CloseDate'] < '2026-04']
val_df = df[(df['CloseDate'] >= '2026-04') & (df['CloseDate'] < '2026-06')]
test_df = df[df['CloseDate'] >= '2026-06']

train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)